# 🏆 Customer Lifetime Value (CLTV) Prediction System
## Phase 7: Machine Learning Model Training
**Project**: Fortune 500 E-Commerce CLTV Prediction  
**Author**: Nisarga N  
**Date**: June 2026  
**Dataset**: UCI Online Retail II  
**Objective**: Fit regression models (Linear Regression, Trees, Ensemble Boosting) to forecast next 90-day spend.


In [ ]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

# Add src to system path
sys.path.append(str(Path.cwd().parent))
from src.model_trainer import ModelTrainer
from src.model_evaluator import ModelEvaluator

warnings.filterwarnings('ignore')

DATA_FEATURES = Path.cwd().parent / 'data' / 'features'
MODELS_DIR = Path.cwd().parent / 'models' / 'saved_models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load Feature Matrix & Split Data
We load the engineered customer profiles and split them into training and evaluation sets.


In [ ]:
df_feat = pd.read_csv(DATA_FEATURES / "customer_features.csv")

trainer = ModelTrainer(df_feat, target_col="target_revenue")
X_train, X_test, y_train, y_test, feature_names = trainer.prepare_data(test_size=0.2, random_state=42)


## 2. Train and Tune Models
Iterating over our 6 regression configurations. We run hyperparameter search to find optimal estimators.


In [ ]:
# Train all models with hyperparameter search enabled
results = trainer.train_all_models(X_train, y_train, tune_hyperparams=True)

# Print best parameters
for name, res in results.items():
    print(f"Model: {name} | Best parameters: {res['best_params']}")


## 3. Save Models and Evaluate
Serialize our trained regressors, log details to the registry, and compare performance scores.


In [ ]:
evaluator = ModelEvaluator(y_test, feature_names)

# Evaluate all models
summary_df = evaluator.generate_evaluation_summary(trainer.models, X_test)

# Save each model and update registry
for name, model in trainer.models.items():
    row = summary_df[summary_df["Model"] == name].iloc[0]
    metrics = {"MAE": float(row["MAE"]), "RMSE": float(row["RMSE"]), "R2": float(row["R2"])}
    trainer.save_model(model, name, MODELS_DIR, metrics)

summary_df
